# LlamaIndex Tutorial 1: Basic RAG with Chroma + HuggingFace Embeddings

This notebook demonstrates how to build a basic RAG (Retrieval Augmented Generation) system using LlamaIndex with Chroma vector database and HuggingFace embedding models.

## What you'll learn:
- Setting up LlamaIndex with Groq LLM
- Using HuggingFace embedding models (BGE, Sentence Transformers)
- Creating a Chroma vector store
- Indexing documents
- Querying with RAG

In [ ]:
# Install required packages
!pip install llama-index llama-index-llms-groq llama-index-embeddings-huggingface llama-index-vector-stores-chroma chromadb sentence-transformers python-dotenv

In [1]:
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
import chromadb

# Load environment variables
load_dotenv()

# Get Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Please set GROQ_API_KEY in your .env file")

print("✅ Environment loaded successfully!")

/Users/rushikeshvinodkharche/opt/anaconda3/envs/meet/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/rushikeshvinodkharche/opt/anaconda3/envs/meet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment loaded successfully!


In [2]:
# Initialize Groq LLM
llm = Groq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

# Initialize HuggingFace Embedding Model
# Using BGE-small-en-v1.5 - a popular open-source embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device="cpu"
)

# Set global settings
Settings.llm = llm
Settings.embed_model = embed_model

print("✅ LLM and Embedding model initialized!")
print(f"LLM Model: llama-3.3-70b-versatile")
print(f"Embedding Model: BAAI/bge-small-en-v1.5")

✅ LLM and Embedding model initialized!
LLM Model: llama-3.3-70b-versatile
Embedding Model: BAAI/bge-small-en-v1.5


## Step 2: Create Sample Documents

Let's create some sample documents to work with. In a real scenario, you would load documents from files.

In [3]:
from llama_index.core import Document

# Create sample documents
documents = [
    Document(
        text="Artificial Intelligence (AI) is transforming the way we work and live. Machine learning algorithms can now recognize patterns in data that humans might miss, enabling breakthroughs in healthcare, finance, and technology."
    ),
    Document(
        text="Large Language Models (LLMs) like GPT-4 and LLaMA have revolutionized natural language processing. These models can understand context, generate human-like text, and assist with various tasks from coding to creative writing."
    ),
    Document(
        text="Vector databases are specialized databases designed to store and query high-dimensional vectors. They are essential for RAG systems, enabling fast similarity search over embeddings of documents and queries."
    ),
    Document(
        text="Retrieval Augmented Generation (RAG) combines the power of information retrieval with language generation. By retrieving relevant context from a knowledge base, RAG systems can provide accurate and up-to-date answers."
    ),
    Document(
        text="Embedding models convert text into numerical vectors that capture semantic meaning. Popular embedding models include OpenAI's text-embedding-ada-002, BGE models from BAAI, and Sentence Transformers."
    ),
]

print(f"✅ Created {len(documents)} sample documents")

✅ Created 5 sample documents


## Step 3: Set Up Chroma Vector Store

In [4]:
# Create Chroma client and collection
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("rag_tutorial")

# Create vector store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("✅ Chroma vector store created!")

✅ Chroma vector store created!


## Step 4: Create Vector Store Index

In [5]:
# Create index from documents
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

print("✅ Index created successfully!")

Generating embeddings: 100%|██████████| 5/5 [00:00<00:00,  5.90it/s]

✅ Index created successfully!


## Step 5: Create Query Engine and Query

In [6]:
# Create query engine
query_engine = index.as_query_engine()

# Query the index
query = "What is RAG and how does it work?"
response = query_engine.query(query)

print(f"Query: {query}\n")
print(f"Response:\n{response}\n")
print(f"\nSource Nodes: {len(response.source_nodes)}")
for i, node in enumerate(response.source_nodes, 1):
    print(f"\nSource {i}:")
    print(f"  Score: {node.score:.4f}")
    print(f"  Text: {node.text[:200]}...")

Query: What is RAG and how does it work?

Response:
RAG combines the power of information retrieval with language generation. It works by retrieving relevant context from a knowledge base, which enables it to provide accurate and up-to-date answers. This is made possible by specialized databases that store and query high-dimensional vectors, allowing for fast similarity search over embeddings of documents and queries.


Source Nodes: 2

Source 1:
  Score: 0.6187
  Text: Retrieval Augmented Generation (RAG) combines the power of information retrieval with language generation. By retrieving relevant context from a knowledge base, RAG systems can provide accurate and up...

Source 2:
  Score: 0.4554
  Text: Vector databases are specialized databases designed to store and query high-dimensional vectors. They are essential for RAG systems, enabling fast similarity search over embeddings of documents and qu...


## Step 6: Try Different Queries

In [7]:
queries = [
    "What are vector databases used for?",
    "Explain large language models",
    "How do embedding models work?"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    response = query_engine.query(query)
    print(f"Answer: {response}\n")


Query: What are vector databases used for?
Answer: Vector databases are used to store and query high-dimensional vectors, enabling fast similarity search over embeddings of documents and queries, particularly in RAG systems.


Query: Explain large language models
Answer: Large language models are capable of understanding context, generating human-like text, and assisting with a wide range of tasks, including coding and creative writing. They have revolutionized the field of natural language processing, enabling advanced applications and uses.


Query: How do embedding models work?
Answer: They convert text into numerical vectors that capture semantic meaning.



## Summary

In this tutorial, we learned:
1. ✅ How to set up LlamaIndex with Groq LLM
2. ✅ How to use HuggingFace embedding models
3. ✅ How to create and use Chroma vector store
4. ✅ How to build a basic RAG system
5. ✅ How to query documents using RAG

### Key Takeaways:
- **Chroma** is a lightweight, in-memory vector database perfect for prototyping
- **HuggingFace embeddings** provide high-quality, open-source alternatives to proprietary models
- **RAG** combines retrieval (finding relevant context) with generation (creating answers)
- The query engine handles the entire RAG pipeline automatically

### Next Steps:
- Try different HuggingFace embedding models (e.g., `sentence-transformers/all-MiniLM-L6-v2`)
- Experiment with different chunk sizes and overlap
- Move to Tutorial 2 to learn about Pinecone vector store